<a href="https://colab.research.google.com/github/Oksana0020/Diamonds-game/blob/main/Diamonds_Game.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Diamonds game
(Search tree)

Robinson Crusoe and Friday play diamonds game on desert island using two piles of stones.

At the beginning of the game, the player chooses to play as Robinson or Friday.
The other character is controlled by AI.

##Goal of the Game
Two piles of stones, Pile A and Pile B, are placed on the ground.

Players take turns removing stones.

The game ends when both piles become (0;0)

##Rules:

On each turn a player makes one move:

1. Remove 1 to 10 stones from one pile

2. Remove the same number from both piles from 1 to 5 stones

All moves must follow these limits.


##Ideas
I want to introduce piles evolution gradually

first: small fixed piles which will be fast to test

then: (100, 100)

then: randomized piles from 80–120, generated once at game start and shown to player.

Also each player can receive two Diamond Tokens like special tools found on the island.A player may spend one token to perform a special move (for example this move can mean - remove the same number of stones from both piles 6-10)

Lucky 13 Rule - When Lucky 13 is triggered 5 stones are added to both piles, triggers multiple times during a game, may benefit or harm a player

Life on the island is unpredictable.From time to time, storms affect the piles. A storm occurs after both players have taken 5 turns each (10 total moves, then 20,30...).Storms repeats regularly.When a storm happens 3 stones are added to one random pile

If time allows to add Twisted Mode
Normal Mode:
The player who makes the move that reaches (0, 0) wins

Twisted Mode:
The player who makes the move that reaches (0, 0) loses

Possible add some dialogues in game - each character has a personality and speaks after making moves like commenting moves

Also need to display pile bars and token counts

##Constants and character labels

In [46]:
WIN_VALUE = 1
LOSS_VALUE = -1

CHAR_ROBINSON = "Robinson"
CHAR_FRIDAY = "Friday"

##CurrentBoard with initial rules

Two piles (a,b), stored sorted (a<=b).

Base legal moves:
*   Remove 1-10 stones from one pile
*   Remove 1-5 stoens from both piles (same number)

In [47]:
class CurrentBoard:
    def __init__(self, a=5, b=8):
        a = int(a); b = int(b)
        self.a, self.b = (a, b) if a <= b else (b, a)
        self.state = self.state_of_play()

    def display_board(self, game_display=False):
        print(f"Piles: ({self.a}, {self.b})")

    def state_of_play(self):
        return "T" if (self.a == 0 and self.b == 0) else "U"

    def other_player(self, forPlayer):
        return "O" if forPlayer == "X" else "X"

    def all_possible_moves(self, forPlayer="X"):
        moves = []

        # remove from pile a (max 10)
        for k in range(1, min(10, self.a) + 1):
            moves.append(CurrentBoard(self.a - k, self.b))

        # remove from pile b (max 10)
        for k in range(1, min(10, self.b) + 1):
            moves.append(CurrentBoard(self.a, self.b - k))

        # remove same from both (max 5)
        for k in range(1, min(5, self.a) + 1):
            moves.append(CurrentBoard(self.a - k, self.b - k))

        # remove duplicates
        unique, seen = [], set()
        for m in moves:
            key = (m.a, m.b)
            if key not in seen:
                seen.add(key)
                unique.append(m)

        return unique

##Checking count of moves and sample results.

In [48]:
cb = CurrentBoard(5, 8)
cb.display_board()
kids = cb.all_possible_moves()
print("Number of legal moves:", len(kids))
print("First 12 moves:", [(k.a, k.b) for k in kids[:12]])

Piles: (5, 8)
Number of legal moves: 17
First 12 moves: [(4, 8), (3, 8), (2, 8), (1, 8), (0, 8), (5, 7), (5, 6), (5, 5), (4, 5), (3, 5), (2, 5), (1, 5)]


##Add minimax tree (SearchTreeNode)

**Each node stores:**

the current board position

the player to move

all possible child positions

A terminal position is a leaf of the tree:

(0,0) means no legal moves remain therefore the player which has to move loses

**The minimax rule is :**

if at least one child is losing for the next player, the current node is winning

otherwise the current node is losing

So:

+1 means the current player can force a win

-1 means the current player will lose with perfect play

In [49]:
class SearchTreeNode:
    def __init__(self, current_board, forPlayer):
        self.board = current_board
        self.forPlayer = forPlayer
        self.children = []
        self.value = None
        self.value_is_assigned = False

        # terminal position where player to move loses
        if self.board.state == "T":
            self.value = LOSS_VALUE
            self.value_is_assigned = True
            return

        # otherwise build all children
        next_player = self.board.other_player(forPlayer)
        for next_board in self.board.all_possible_moves(forPlayer):
            child = SearchTreeNode(next_board, next_player)
            self.children.append(child)

    def min_max_value(self):
        if self.value_is_assigned:
            return self.value

        # if any child is losing for the next player current position is winning
        for child in self.children:
            if child.min_max_value() == LOSS_VALUE:
                self.value = WIN_VALUE
                self.value_is_assigned = True
                return self.value

        # otherwise all children are winning for next player
        self.value = LOSS_VALUE
        self.value_is_assigned = True
        return self.value

##Checking minimax value on a small starting position

In [41]:
cb = CurrentBoard(5, 8)
st = SearchTreeNode(cb, "X")

print("Root board:", (cb.a, cb.b))
print("Children count:", len(st.children))
print("Minimax value:", st.min_max_value())

Root board: (5, 8)
Children count: 17
Minimax value: 1


##Choose the best next move

Selecting the best child:

if a child has value LOSS_VALUE, that means the next player loses there

so this is a winning move for the current player

if no such child exists, return the first child

In [50]:
def best_child_board(node: SearchTreeNode):
    if not node.children:
        return None
    for child in node.children:
        if child.value == LOSS_VALUE:
            return child.board
    return node.children[0].board

##Checking best move selection

In [54]:
cb = CurrentBoard(5, 8)
st = SearchTreeNode(cb, "X")
st.min_max_value()
best = best_child_board(st)

print("Current board:", (cb.a, cb.b))
print("Best next board:", (best.a, best.b) if best else None)

Current board: (5, 8)
Best next board: (3, 5)


##Validate player moves if they are legal
Before creating a playable loop I need a helper that checks whether a new board is a legal result of the current board. This will be used to verify human input

generate all possible next boards

check if (a,b) appears among them

In [55]:
def is_legal_move(current: CurrentBoard, nxt: CurrentBoard) -> bool:
    return any((m.a, m.b) == (nxt.a, nxt.b) for m in current.all_possible_moves())

testing one legal and one illegal move from the same starting position.

In [56]:
cb = CurrentBoard(5, 8)
legal_test = CurrentBoard(5, 3)
illegal_test = CurrentBoard(1, 1)

print("Legal move test:", is_legal_move(cb, legal_test))
print("Illegal move test:", is_legal_move(cb, illegal_test))

Legal move test: True
Illegal move test: False


Add basic play loop